**Lab5 Ex3 By Prajakta Ghorpade SCFP125004**


Exercise 3 — Interview Transcript to Candidate Scorecard Pipeline
Build a 3-step pipeline. Step 1 extracts the candidate's answers to each interview question from a raw transcript. Step 2 evaluates each answer against a given set of skills, flagging any skill that wasn't clearly demonstrated. Step 3 formats the evaluation into a structured hiring scorecard with an overall recommendation.

Input: transcript_text, skills_to_assess

Output: the extracted answers from Step 1, the flagged skill evaluation from Step 2, and the final scorecard from Step 3

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (field: list of question + answer pairs)

Step 2 — Template: ChatPromptTemplate (system persona: "technical interviewer evaluating skill evidence")  |  Parser: PydanticOutputParser (fields: skill, demonstrated: bool, evidence_or_gap)

Step 3 — Template: ChatPromptTemplate (system persona: "hiring panel coordinator")  |  Parser: PydanticOutputParser (fields: list of scorecard rows + overall_recommendation)



Install packages

In [2]:
%pip install -q langchain langchain-core langchain-groq pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.0 MB/s eta 0:00:00


Imports and Groq setup

In [3]:
import os
from typing import List

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

llm.invoke("hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. The instruction: "You are ChatGPT, a large language model trained by OpenAI." There\'s no special instruction. Just respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 72, 'total_tokens': 129, 'completion_time': 0.063267098, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.003366831, 'prompt_tokens_details': None, 'queue_time': 0.127675389, 'total_time': 0.066633929}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16736-eea9-7d31-9b44-20cdaa6e6498-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 57, 'total_tokens': 129, 'output_token_details': {'reasoning': 39}})

Input Transcript and Skills

In [4]:
transcript_text = """
Interviewer: Tell me about a project where you used Python.
Candidate: I used Python and Pandas to clean a CSV dataset,
remove duplicate rows, and calculate summary statistics.

Interviewer: How do you work with SQL?
Candidate: I have written SELECT queries with WHERE and GROUP BY
to summarize sales records. I have not used window functions yet.

Interviewer: Describe your experience with teamwork.
Candidate: In my college project, I divided tasks with two teammates
and gave updates during our weekly meetings.
"""

skills_to_assess = [
    "Python",
    "SQL",
    "Data cleaning",
    "Teamwork",
    "Cloud deployment"
]

print("Interview transcript loaded successfully.")
print("\nSkills to assess:")
print(skills_to_assess)

Interview transcript loaded successfully.

Skills to assess:
['Python', 'SQL', 'Data cleaning', 'Teamwork', 'Cloud deployment']


Extract Questions and Answers

In [5]:
class QuestionAnswer(BaseModel):
    question: str
    answer: str

class ExtractedAnswers(BaseModel):
    answers: List[QuestionAnswer]


extract_parser = PydanticOutputParser(
    pydantic_object=ExtractedAnswers
)

extract_prompt = PromptTemplate(
    template="""Extract every interview question and its
corresponding candidate answer from the transcript.
Use only the transcript. Do not invent missing answers.
If an answer is missing, write "Not provided".

Transcript:
{transcript_text}

{format_instructions}
""",
    input_variables=["transcript_text"],
    partial_variables={
        "format_instructions": extract_parser.get_format_instructions()
    }
)

extract_chain = extract_prompt | llm | extract_parser

step1_output = extract_chain.invoke({
    "transcript_text": transcript_text
})

print(step1_output.model_dump_json(indent=2))

{
  "answers": [
    {
      "question": "Tell me about a project where you used Python.",
      "answer": "I used Python and Pandas to clean a CSV dataset, remove duplicate rows, and calculate summary statistics."
    },
    {
      "question": "How do you work with SQL?",
      "answer": "I have written SELECT queries with WHERE and GROUP BY to summarize sales records. I have not used window functions yet."
    },
    {
      "question": "Describe your experience with teamwork.",
      "answer": "In my college project, I divided tasks with two teammates and gave updates during our weekly meetings."
    }
  ]
}


Skill Evaluation

In [6]:
class SkillEvaluation(BaseModel):
    skill: str
    demonstrated: bool
    evidence_or_gap: str


class SkillEvaluations(BaseModel):
    evaluations: List[SkillEvaluation]


skill_parser = PydanticOutputParser(
    pydantic_object=SkillEvaluations
)

skill_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a technical interviewer evaluating skill evidence.
Evaluate each requested skill using only the extracted answers.
Set demonstrated=True only when clear evidence is present.
Otherwise set demonstrated=False and explain the gap.
Return one evaluation for every requested skill.

{format_instructions}"""
    ),
    (
        "human",
        """Skills to assess:
{skills_to_assess}

Extracted answers:
{extracted_answers}"""
    )
]).partial(
    format_instructions=skill_parser.get_format_instructions()
)

skill_chain = skill_prompt | llm | skill_parser

step2_output = skill_chain.invoke({
    "skills_to_assess": ", ".join(skills_to_assess),
    "extracted_answers": step1_output.model_dump_json(indent=2)
})

print(step2_output.model_dump_json(indent=2))

{
  "evaluations": [
    {
      "skill": "Python",
      "demonstrated": true,
      "evidence_or_gap": "Used Python and Pandas to clean a CSV dataset and calculate summary statistics."
    },
    {
      "skill": "SQL",
      "demonstrated": true,
      "evidence_or_gap": "Wrote SELECT queries with WHERE and GROUP BY clauses to summarize sales records."
    },
    {
      "skill": "Data cleaning",
      "demonstrated": true,
      "evidence_or_gap": "Cleaned a CSV dataset by removing duplicate rows and computing summary statistics."
    },
    {
      "skill": "Teamwork",
      "demonstrated": true,
      "evidence_or_gap": "Divided tasks with two teammates and provided updates during weekly meetings."
    },
    {
      "skill": "Cloud deployment",
      "demonstrated": false,
      "evidence_or_gap": "No evidence of cloud deployment experience was provided."
    }
  ]
}


Final Hiring Scorecard

In [7]:
class ScorecardRow(BaseModel):
    skill: str
    status: str
    evidence_or_gap: str


class HiringScorecard(BaseModel):
    scorecard_rows: List[ScorecardRow]
    overall_recommendation: str


scorecard_parser = PydanticOutputParser(
    pydantic_object=HiringScorecard
)

scorecard_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a hiring panel coordinator.
Create a scorecard with one row per evaluated skill.
Use "Demonstrated" only for demonstrated=True.
Otherwise use "Not clearly demonstrated".
Base the recommendation only on the supplied evidence.
If evidence is incomplete, recommend further assessment.
Do not invent evidence or qualifications.

{format_instructions}"""
    ),
    (
        "human",
        """Skill evaluation:
{skill_evaluation}"""
    )
]).partial(
    format_instructions=scorecard_parser.get_format_instructions()
)

scorecard_chain = scorecard_prompt | llm | scorecard_parser

step3_output = scorecard_chain.invoke({
    "skill_evaluation": step2_output.model_dump_json(indent=2)
})

print(step3_output.model_dump_json(indent=2))

{
  "scorecard_rows": [
    {
      "skill": "Python",
      "status": "Demonstrated",
      "evidence_or_gap": "Used Python and Pandas to clean a CSV dataset and calculate summary statistics."
    },
    {
      "skill": "SQL",
      "status": "Demonstrated",
      "evidence_or_gap": "Wrote SELECT queries with WHERE and GROUP BY clauses to summarize sales records."
    },
    {
      "skill": "Data cleaning",
      "status": "Demonstrated",
      "evidence_or_gap": "Cleaned a CSV dataset by removing duplicate rows and computing summary statistics."
    },
    {
      "skill": "Teamwork",
      "status": "Demonstrated",
      "evidence_or_gap": "Divided tasks with two teammates and provided updates during weekly meetings."
    },
    {
      "skill": "Cloud deployment",
      "status": "Not clearly demonstrated",
      "evidence_or_gap": "No evidence of cloud deployment experience was provided."
    }
  ],
  "overall_recommendation": "Recommend further assessment"
}


Display All Three Steps

In [8]:
print("STEP 1 — EXTRACTED QUESTIONS AND ANSWERS")
print(step1_output.model_dump_json(indent=2))

print("\nSTEP 2 — SKILL EVALUATION")
print(step2_output.model_dump_json(indent=2))

print("\nSTEP 3 — FINAL HIRING SCORECARD")
print(step3_output.model_dump_json(indent=2))

STEP 1 — EXTRACTED QUESTIONS AND ANSWERS
{
  "answers": [
    {
      "question": "Tell me about a project where you used Python.",
      "answer": "I used Python and Pandas to clean a CSV dataset, remove duplicate rows, and calculate summary statistics."
    },
    {
      "question": "How do you work with SQL?",
      "answer": "I have written SELECT queries with WHERE and GROUP BY to summarize sales records. I have not used window functions yet."
    },
    {
      "question": "Describe your experience with teamwork.",
      "answer": "In my college project, I divided tasks with two teammates and gave updates during our weekly meetings."
    }
  ]
}

STEP 2 — SKILL EVALUATION
{
  "evaluations": [
    {
      "skill": "Python",
      "demonstrated": true,
      "evidence_or_gap": "Used Python and Pandas to clean a CSV dataset and calculate summary statistics."
    },
    {
      "skill": "SQL",
      "demonstrated": true,
      "evidence_or_gap": "Wrote SELECT queries with WHERE and 